# Python Mutability: Cookie Jars and Labels

*Why changing one variable sometimes changes another one too, and the six everyday bugs that come out of it.*

**What you'll learn**

1. What a variable really is in Python (and what it isn't)
2. Mutable vs immutable objects, and how to check which is which using `id()`
3. Six traps that mutability sets for you, each explained as The Trap, Why It Happens, and The Fix
4. How mutability changes what happens when you pass data into a function

## The Story: A Cookie Jar With Two Labels

Imagine a kitchen shelf lined with cookie jars.

- The **cookies** inside a jar are the **data**.
- The **jar** is a block of memory in your RAM, the place where the data actually lives.
- The **label** stuck on the outside of the jar is the **variable name**.

Say we bake some chocolate cookies, put them in a jar and stick a label on it that says **"chocolate cookies"**. That's exactly what Python does when we write:

```python
chocolate_cookies = ["cookie", "cookie", "cookie"]
```

Python first makes the jar and fills it (creates the list in memory), and then sticks the label `chocolate_cookies` on it.

Now, these cookies also happen to be gluten free. So we stick a **second** label on the same jar: **"gluten free"**.

```python
gluten_free = chocolate_cookies
```

We didn't bake a new batch and we didn't buy a new jar. There is still **one jar**, it just has **two labels** on it now. If someone grabs a cookie from the jar labelled "gluten free", the jar labelled "chocolate cookies" has one less cookie too, because it's the same jar.

That's the whole idea behind this article: **a variable is a label on a jar, not the jar itself.**

Jars also come in two kinds:

- **Open jars (mutable):** you can open the lid, take cookies out, put more in. Everyone looking at that jar, through whichever label, sees the new contents.
- **Sealed jars (immutable):** once filled and sealed, the contents can never change. If you want different cookies, Python has to fill a **new** jar, seal it, and move *your* label over to it. Any other labels stay on the old jar.

Almost every bug in this article comes down to the same thing: **two labels on one open jar**, when you thought you had two jars.

## The Definition

- **Mutable:** an object that can be changed after it is created.
- **Immutable:** an object that can never be changed after it is created. Any "edit" produces a brand-new object.

Key vocabulary:

| Term | Meaning in our story |
|---|---|
| **Object** | The jar with its cookies, sitting in memory |
| **Name / variable** | The label on the jar |
| **`id(obj)`** | The jar's identity. In CPython (the standard Python), it's the memory address where the jar sits. |
| **Mutate** | Open the jar and change what's inside |
| **Rebind** | Peel a label off and stick it on a different jar |

> **The one idea to remember:** *mutate* vs *rebind*. Mutating changes what's inside the jar, so every label on that jar sees the change. Rebinding moves one label to another jar, and nobody else is affected.

## 1. Variables Are Labels

Let's put the cookie jar story into code. One jar, two labels. Then someone eats a cookie using the "gluten free" label.

In [1]:
chocolate_cookies = ["cookie", "cookie", "cookie"]
gluten_free = chocolate_cookies

gluten_free.pop()

print("chocolate_cookies:", chocolate_cookies)
print("gluten_free:      ", gluten_free)
print(id(chocolate_cookies) == id(gluten_free))

chocolate_cookies: ['cookie', 'cookie']
gluten_free:       ['cookie', 'cookie']
True


We only touched `gluten_free`, but `chocolate_cookies` lost a cookie too. Same jar, same id.

The same thing happens with a dict of friends that we give a second name:

In [2]:
friends = {"Jon": 1, "Ron": 2, "Amen": 3}
friends_dict = friends

print(id(friends))
print(id(friends_dict))
print(friends is friends_dict)

140615629305920
140615629305920
True


The ids are identical. There is **one** dict with **two** labels on it.

The data is the dictionary (the jar and its cookies). `friends` is just the label we stuck on it.

> **Note:** your id numbers will differ from the ones shown here. Memory addresses change on every run and on every machine. What matters is whether two ids are *equal*, not what the number is.

## 2. Same Contents Doesn't Mean Same Object

What if we build a dict with the exact same contents again, under the same name?

In [3]:
friends = {"Jon": 1, "Ron": 2, "Amen": 3}
print(id(friends))

friends = {"Jon": 1, "Ron": 2, "Amen": 3}
print(id(friends))

140615629302080
140615629302144


The ids are different. Same name, same type, same values, but **two separate objects**.

**What happens under the hood?** Python runs `friends = {...}` in this order:

1. **Evaluate the right-hand side first.** Fill a brand new jar (create the new dict in memory).
2. **Then** peel the `friends` label off the old jar and stick it on the new one.
3. Only now can the old jar be thrown away (if no other label is still on it).

At step 1 the old jar still exists and is still sitting in its spot, so the new jar *has* to go somewhere else. **Two objects that are alive at the same time can never share an id.** The spot is taken.

> **Gotcha: ids get recycled.** An `id` is unique only among objects that are alive *at the same moment*. Once a jar is thrown away, its spot on the shelf can be reused by a new jar. So *different ids* proves two different objects. But *the same id at two different moments* proves nothing.

In the cell below, each temporary dict is thrown away as soon as `id()` has read it. That means the second dict often lands in the spot the first one just freed:

In [4]:
print(id({"Jon": 1}) == id({"Jon": 1}))

True


## 3. Mutating in Place

Now let's change a value *inside* the dict.

In [5]:
friends = {"Jon": 1, "Ron": 2, "Amen": 3}
print(id(friends))

friends["Amen"] = 10
print(id(friends))
print(friends)

140615629303040
140615629303040
{'Jon': 1, 'Ron': 2, 'Amen': 10}


The id didn't change. `friends["Amen"] = 10` did **not** create a new dict. We opened the existing jar, took one thing out and put something else in. In other words, we **mutated** the dict.

That's what it means for `dict` to be a **mutable** type: it can be modified after it's created.

## 4. Immutable Types: The Sealed Jars

The immutable types you'll use every day are **int, float, str, and tuple**. The complete list of built-ins is a bit longer:

| Immutable (sealed jar) | Mutable (open jar) |
|---|---|
| `int`, `float`, `complex`, `bool` | `list` |
| `str` | `dict` |
| `tuple` | `set` |
| `frozenset` | `bytearray` |
| `bytes` | Instances of your own classes (by default) |
| `None` | |

Any "edit" to an immutable object gives you back a **new** object.

In [6]:
int_1 = 1
print(id(int_1))

int_1 = int_1 + 1
print(id(int_1))
print(int_1)

20100672
20100704
2


The `1` object can't turn into `2`. So `int_1 + 1` produces a different int object, `2`, and the `int_1` label moves onto it. The original `1` is untouched.

Strings work the same way. Notice that there isn't even a way to edit a string in place:

In [7]:
name = "vaibhav"
print(id(name))

name = name.upper()
print(id(name), name)

try:
    name[0] = "v"
except TypeError as e:
    print("TypeError:", e)

140615629316064
140615629316112 VAIBHAV
TypeError: 'str' object does not support item assignment


`.upper()` doesn't change the string. It *returns a new one*, and we rebind `name` to it. Trying to assign to a character fails outright, because you can't open a sealed jar.

> **Under the hood: the small integer cache.** CPython creates the integers **-5 to 256** once, at startup, and reuses them everywhere. So when we computed `1 + 1`, the `2` object already existed, and `int_1` was simply moved onto that shared object. This is a CPython optimisation, not a language rule. That's exactly why you should never use `is` to compare numbers (see Trap 5).

> **Gotcha: an immutable container can hold mutable things.** A tuple can't change *which* objects it holds. But if one of those objects is a list, the list itself can still change.

In [8]:
record = ("Jon", [1, 2])
record[1].append(3)
print(record)

('Jon', [1, 2, 3])


The tuple is unchanged: its second slot still points to the same list object. The *list* is what mutated. "Immutable" means the tuple's slots are fixed, **not** that everything reachable from it is frozen. This is also why a tuple that contains a list can't be used as a dict key: it isn't hashable.

### Trap 1: Aliasing

**The Trap**

We have a list `numb1`, and we "copy" it into `numb2`. Then we change `numb2`.

In [9]:
numb1 = [1, 2, 3]
numb2 = numb1

numb2.append(10)

print("numb1:", numb1)
print("numb2:", numb2)
print(id(numb1) == id(numb2))

numb1: [1, 2, 3, 10]
numb2: [1, 2, 3, 10]
True


**Why It Happens**

`numb2 = numb1` makes a new **label**, not a new list. It's the chocolate cookies and gluten free story again: two labels, one jar. Appending through `numb2` writes on the only list that exists, so `numb1` "changes" too. When two names share one object, that's called **aliasing**.

**The Fix**

Ask for a copy explicitly. Any of these three works:

In [10]:
numb1 = [1, 2, 3]
numb2 = numb1.copy()
numb3 = numb1[:]
numb4 = list(numb1)

numb2.append(10)

print("numb1:", numb1)
print("numb2:", numb2)
print(numb1 is numb2, numb1 is numb3, numb1 is numb4)

numb1: [1, 2, 3]
numb2: [1, 2, 3, 10]
False False False


Each of these builds a **new** outer list. For dicts, use `d.copy()` or `dict(d)`.

> **Heads up:** all of these are **shallow** copies. Trap 4 shows when that's not enough.

### Trap 2: Shared Class Attribute

**The Trap**

A `Team` class with a `members` list. We create two teams and add players to each.

In [11]:
class Team:
    members = []

    def __init__(self, name):
        self.name = name

    def add(self, person):
        self.members.append(person)


india = Team("india")
pak = Team("pak")

india.add("virat")
india.add("sachin")
india.add("vaibhav")

pak.add("afridi")
pak.add("wasim")
pak.add("akram")

print("india:", india.members)
print("pak:  ", pak.members)

india: ['virat', 'sachin', 'vaibhav', 'afridi', 'wasim', 'akram']
pak:   ['virat', 'sachin', 'vaibhav', 'afridi', 'wasim', 'akram']


Both teams have all six players. We used two different instances, but they were reading and updating the **same list**.

**Why It Happens**

`members = []` sits in the **class body**, so it is created **once**, when the class is defined. It belongs to the class `Team` itself, not to each team.

When you write `self.members`, Python:

1. Looks for `members` on the instance (`india`). It doesn't find it.
2. Falls back to the class (`Team`). It finds the one shared list there.
3. `.append()` then **mutates** that shared list.

In [12]:
print(india.members is pak.members is Team.members)
print(india.__dict__)

True
{'name': 'india'}


The instance's own attribute dictionary has only `name`. There's no `members` in it at all.

> **Subtle twist:** `self.members = ["x"]` (an *assignment*) would **not** touch the shared list. It would create a new instance attribute that hides the class one. Mutating leaks to every instance, while rebinding doesn't. It's the mutate-vs-rebind distinction again.

**The Fix**

Create the list inside `__init__`, so each instance gets its own list on every construction.

In [13]:
class Team:
    def __init__(self, name):
        self.name = name
        self.members = []

    def add(self, person):
        self.members.append(person)


india = Team("india")
pak = Team("pak")

india.add("virat")
india.add("sachin")
india.add("vaibhav")

pak.add("afridi")
pak.add("wasim")
pak.add("akram")

print("india:", india.members)
print("pak:  ", pak.members)

india: ['virat', 'sachin', 'vaibhav']
pak:   ['afridi', 'wasim', 'akram']


> **Pro tip:** class attributes are fine for **immutable** values that really are shared, such as `sport = "cricket"` or a constant. Keep **mutable, per-object** state in `__init__`.

### Trap 3: Mutable Default Arguments (The Classic Trap)

This is a close cousin of Trap 2. If you use a mutable default argument in `__init__` (or in any function), every call that relies on the default shares the **exact same object**.

**The Trap**

In [14]:
class Player:
    def __init__(self, name, inventory=[]):
        self.name = name
        self.inventory = inventory


player1 = Player("Alice")
player2 = Player("Bob")

player1.inventory.append("Sword")

print(player2.inventory)

['Sword']


Alice finds a sword, and Bob magically has it too.

**Why It Happens**

Default values are evaluated **once**, when the `def` line runs (here, when the class is defined). They are **not** evaluated each time the function is called. Python stores that single list on the function object and hands the same list to every call that doesn't pass its own.

You can see the stored default directly:

In [15]:
print(Player.__init__.__defaults__)
print(player1.inventory is player2.inventory is Player.__init__.__defaults__[0])

(['Sword'],)
True


Alice and Bob each got a label on the **one** list (jar) that lives inside the function's defaults. That's also why the bug "remembers" items across calls.

**Fix 1 (the standard idiom): use `None` as a placeholder**

In [16]:
class Player:
    def __init__(self, name, inventory=None):
        self.name = name
        self.inventory = [] if inventory is None else inventory


player1 = Player("Alice")
player2 = Player("Bob")

player1.inventory.append("Sword")

print("Alice:", player1.inventory)
print("Bob:  ", player2.inventory)

Alice: ['Sword']
Bob:   []


`None` is immutable, so sharing it is harmless. The fresh `[]` is built *inside the function body*, and the body runs on every call. Notice that we test with `is None`, not `== None`. That's Trap 5.

**Fix 2: copy whatever comes in (a defensive copy)**

Another option is to rebind a fresh list for each instance. The most robust way to do that is to *copy* the argument inside `__init__`:

In [17]:
class Player:
    def __init__(self, name, inventory=()):
        self.name = name
        self.inventory = list(inventory)


shared_loot = ["Shield"]
player1 = Player("Alice", shared_loot)
player2 = Player("Bob")

player1.inventory.append("Sword")

print("shared_loot:", shared_loot)
print("Alice:      ", player1.inventory)
print("Bob:        ", player2.inventory)

shared_loot: ['Shield']
Alice:       ['Shield', 'Sword']
Bob:         []


The default is an immutable empty tuple `()`, and `list(...)` makes a new list. Every player gets their own list, **and** the caller's list (`shared_loot`) is protected. With Fix 1, Alice's inventory would still be an alias of `shared_loot`.

> **Note:** setting `player2.inventory = []` *after* you create the object also works, but you have to remember to do it every time. That's fragile.

> **Pro tip, using dataclasses:** use `field(default_factory=list)`, which calls `list()` once per instance. Python actually raises a `ValueError` if you write `inventory: list = []` in a dataclass, specifically to stop this trap.

In [18]:
from dataclasses import dataclass, field


@dataclass
class Player:
    name: str
    inventory: list = field(default_factory=list)


player1 = Player("Alice")
player2 = Player("Bob")

player1.inventory.append("Sword")

print("Alice:", player1.inventory)
print("Bob:  ", player2.inventory)

Alice: ['Sword']
Bob:   []


### Trap 4: Shallow vs Deep Copy

Back to the kitchen. Say you have a big jar, and inside it are smaller packets of cookies. Python lists don't really hold the packets themselves; they hold slips of paper saying *which packet* goes where. A **shallow copy** gives you a new big jar with copies of those slips, so both jars still point to the *same* packets. A **deep copy** makes new packets with new cookies too.

**The Trap**

If a list contains inner lists, `.copy()` (the fix from Trap 1) creates only a new **outer** list. The inner lists are still shared.

In [19]:
grid = [[0, 0], [0, 0]]
g2 = grid.copy()

g2[0][0] = 9

print("grid:", grid)
print("g2:  ", g2)

grid: [[9, 0], [0, 0]]
g2:   [[9, 0], [0, 0]]


We changed `g2`, and the change shows up in the original `grid`.

**Why It Happens**

`.copy()` builds a new outer list and fills it with references to the **same** inner list objects. It copies only one level deep.

In [20]:
grid = [[0, 0], [0, 0]]
g2 = grid.copy()

print(grid is g2)
print(grid[0] is g2[0])

False
True


The big jars are different, but the packets inside are the same.

**The Fix**

Use `copy.deepcopy`, which copies everything, inner lists included.

In [21]:
import copy

grid = [[0, 0], [0, 0]]
g3 = copy.deepcopy(grid)

g3[0][0] = 15

print("grid:", grid)
print("g3:  ", g3)
print(grid[0] is g3[0])

grid: [[0, 0], [0, 0]]
g3:   [[15, 0], [0, 0]]
False


`copy.deepcopy` walks the whole structure recursively and copies every mutable object it finds. It also handles structures that refer to themselves. It's slower and uses more memory, so save it for **nested mutable data that you plan to modify independently**.

> **A related trap: building a grid with `*`.** In `[[0] * 2] * 3`, the outer `* 3` repeats a reference to **one** inner list three times. Build it with a comprehension instead.

In [22]:
bad = [[0] * 2] * 3
bad[0][0] = 9
print("bad: ", bad)

good = [[0] * 2 for _ in range(3)]
good[0][0] = 9
print("good:", good)

bad:  [[9, 0], [9, 0], [9, 0]]
good: [[9, 0], [0, 0], [0, 0]]


### Trap 5: `is` vs `==`

This follows straight on from the `id()` work above.

- `==` asks: **"Do these hold equal values?"**
- `is` asks: **"Are these the very same object?"** (Do they have the same `id`?)

Think of two jars on the shelf, each with exactly the same cookies inside. The contents are equal (`==` is `True`), but they are two different jars (`is` is `False`).

**The Trap**

Using `is` to compare values. It *sometimes* works by accident, and that's what makes it dangerous.

In [23]:
a = [1, 2]
b = [1, 2]
print(a == b)
print(a is b)

x = int("1000")
y = int("1000")
print(x == y)
print(x is y)

True
False
True
False


**Why It Happens**

`is` compares identity, not value. Two equal lists are still two separate objects. With numbers and short strings it's sneakier: because of caching (remember the small-integer cache?), `is` can return `True` for small values in your test and then `False` for larger values in production. Python 3.8+ even shows a `SyntaxWarning` when you write `is` against a literal.

**The Fix**

- Use `==` whenever you care about **values**.
- Use `is` only for **singletons**, meaning objects that exist exactly once. The most common one is `None`: `if x is None:`. `True`, `False`, and custom sentinel objects also count.

That's exactly the check the default-argument fix in Trap 3 relies on. There is only ever one `None` object, so identity is the correct and fastest test.

### Trap 6: `+=` Behaves Differently on Mutable and Immutable Types

**The Trap**

In [24]:
j = [1, 2]
y = j

y += [10]

print("j:", j)
print("y:", y)
print(j is y)

j: [1, 2, 10]
y: [1, 2, 10]
True


`y += [10]` asked the existing list to add `10` to itself. It's still the same list, so `j` sees the change too.

Now compare that with the long form:

In [25]:
j = j + [29, 30]

print("j:", j)
print("y:", y)
print(j is y)

j: [1, 2, 10, 29, 30]
y: [1, 2, 10]
False


`j + [29, 30]` **built a brand-new list**, `[1, 2, 10, 29, 30]`, and *then* moved the `j` label onto it. The old list `[1, 2, 10]` still exists, and `y` still points to it.

`+=` and `= ... +` look interchangeable, but on a list they are not.

**Why It Happens (under the hood)**

`x += v` first tries `x.__iadd__(v)`, which means "add in place":

| Type | Has `__iadd__`? | What `x += v` does |
|---|---|---|
| `list` | Yes | Extends the **existing** list (like `.extend()`), then rebinds `x` to that same object. **Net effect: mutation.** |
| `int`, `str`, `tuple` | No (they can't change) | Falls back to `x = x + v`. **A new object, and only `x` is rebound.** |

`x = x + v` always calls `__add__`, which always returns a **new** object.

In [26]:
n = 5
m = n

n += 1

print(n, m, n is m)

6 5 False


On an int, `+=` left `m` alone because a new object was created.

**The Fix**

Decide which one you want, and write that explicitly:

- **You want a new list and other aliases left untouched?** Use `x = x + [...]`, or copy first.
- **You want an in-place update?** Use `+=` or `.extend()`, and make sure nothing else depends on the old contents.

> **A pro-level gotcha: a tuple, a list and `+=`**

In [27]:
t = ([1, 2],)

try:
    t[0] += [3]
except TypeError as e:
    print("TypeError:", e)

print(t)

TypeError: 'tuple' object does not support item assignment
([1, 2, 3],)


It raised an error **and** the list was changed anyway. Here's how:

1. `t[0].__iadd__([3])` mutates the list in place. This succeeds.
2. Python then tries to store the result back with `t[0] = ...`. Tuples don't allow item assignment, so you get a `TypeError`.

By then the mutation has already happened. Write `t[0].extend([3])` instead.

## 6. Mutability and Function Arguments

One more trip to the kitchen. When you call a function, Python doesn't bake a fresh batch of your cookies, and it doesn't hand over your own label either. It sticks a **new, temporary label** (the parameter name) on the **same jar**.

This is called **call by sharing**, or *pass by object reference*. It leads to two rules:

- If the function **mutates** the object, the caller **sees the change**.
- If the function **rebinds** its parameter, only the function's own temporary label moves. The caller **sees nothing**.

### 6.1 A Function That Mutates a Dict

Here's a dict of friends and how long ago we last saw each of them. First, let's record the ids:

In [28]:
friends = {"jon": 21, "mon": 22, "gon": 23}

jon_id_before = id(friends["jon"])

print(id(friends))
print(id(friends["jon"]))
print(id(friends["mon"]))
print(id(friends["gon"]))

140615629379584
20101312
20101344
20101376


This function resets the "last seen" value to 0. It also prints the id of the dict it receives, so we can compare ids inside and outside the function.

In [29]:
def see_friends(friend_list, friend_name):
    print("inside: ", id(friend_list))
    friend_list[friend_name] = 0


print("before: ", id(friends))
see_friends(friends, "jon")
print("after:  ", id(friends))
print(friends)

before:  140615629379584
inside:  140615629379584
after:   140615629379584
{'jon': 0, 'mon': 22, 'gon': 23}


The id is the same inside and outside. The function's `friend_list` was just another label on the caller's dict. The dict was edited, and its id never changed.

**The Trap:** a function that **silently changes its input**. If any other part of the program holds a label on this dict, `jon` is now 0 there too, whether or not that code expected it.

What about the value *inside* the dict?

In [30]:
print(jon_id_before)
print(id(friends["jon"]))

20101312
20100640


The value under `"jon"` **does** have a new id: `21` and `0` are two different int objects. The dict's `"jon"` slot was **rebound** to the `0` object, while the dict itself kept its identity.

> **Mutating a container means rebinding the slots inside it.** The container keeps its id, and its contents point somewhere new.

**Why It Happens:** the parameter is an alias for the caller's object (call by sharing).

**The Fix:** pick one of these two styles, and make the choice obvious.

- **Mutate on purpose:** name and document the function as modifying its input, and return `None`. This is how `list.sort()` behaves.
- **Don't mutate:** copy the input, change the copy, and return it.

In [31]:
def see_friend(friend_list, friend_name):
    updated = dict(friend_list)
    updated[friend_name] = 0
    return updated


friends = {"jon": 21, "mon": 22, "gon": 23}
new_friends = see_friend(friends, "mon")

print("friends:    ", friends)
print("new_friends:", new_friends)

friends:     {'jon': 21, 'mon': 22, 'gon': 23}
new_friends: {'jon': 21, 'mon': 0, 'gon': 23}


> **Python's own convention:** methods that mutate in place return `None` (`list.sort()`, `list.append()`, `dict.update()`). Functions that return a new object leave their input alone (`sorted()`, `str.upper()`). Following this convention tells readers which kind of function they're calling.

### 6.2 A Function That "Changes" an Int

**The Trap**

In [32]:
age = 20


def increase_age(user_age):
    user_age = user_age + 1


print(id(age))
increase_age(age)
print(id(age))
print(age)

20101280
20101280
20


Reading the code, you might expect this: the function runs `age = age + 1`, so `age` becomes 21. And since changing an int gives a new object (Section 4), the second id should be different.

**Neither happens.** `age` is still 20, with the same id.

**Why It Happens**, step by step:

1. **The call:** Python effectively runs `user_age = age`. Both labels are now on the same `20`.
2. `user_age + 1` creates a **new** int object, `21`.
3. `user_age = ...` moves **only** the local `user_age` label onto `21`. `age` is still stuck on `20`.
4. The function ends and the `user_age` label is thrown away. Nothing points to `21` any more, so it gets thrown away too.

What about `+=`?

In [33]:
def increase_age(user_age):
    user_age += 1


print(age)
print(id(age))
increase_age(age)
print(age)
print(id(age))

20
20101280
20
20101280


The result is the same. On an int, `user_age += 1` quietly does the same thing as `user_age = user_age + 1`: ints have no `__iadd__` (Trap 6), so Python creates a new object and rebinds the local label.

**The Fix:** if the caller needs the new value, **return it** and reassign it on the caller's side.

In [34]:
def increase_age(user_age):
    return user_age + 1


age = 20
age = increase_age(age)
print(age)

21


### 6.3 The Same Code With a List

Now swap the int for a list:

In [35]:
ages = [20]


def increase_age(l):
    l += [1]


increase_age(ages)
print(ages)

[20, 1]


**The code looks the same, but the result is the opposite.** The list *did* change, because `+=` on a list calls `__iadd__`, which mutates the shared object.

> **The type decides it:** mutable objects get changed in place, while immutable ones get rebound. Same syntax, different behaviour, depending on which object your label is stuck to.

## 7. Cheat Sheet

| Situation | What actually happens | Safe pattern |
|---|---|---|
| `b = a` | Second label, same object | `b = a.copy()` / `a[:]` / `list(a)` |
| Nested list or dict copied with `.copy()` | Inner objects still shared | `copy.deepcopy(a)` |
| Mutable class attribute (`members = []` in class body) | One list shared by every instance | Create it in `__init__` |
| Mutable default (`def f(x=[])`) | One list shared by every call | `x=None` then `x = [] if x is None else x`, or `field(default_factory=list)` |
| `a is b` to compare values | Compares identity; works only by caching luck | `a == b`; keep `is` for `None` |
| `x += v` on a list | Mutates in place; aliases see it | `x = x + v` if you want a new list |
| `x += v` on int/str/tuple | New object; only `x` rebound | (Nothing to fix, just know it) |
| Function mutates its argument | Caller's object changes | Copy and return, or document it clearly |
| Function rebinds its argument | Caller sees nothing | Return the new value and reassign it |
| `[[0] * n] * m` | Every row is the same list | `[[0] * n for _ in range(m)]` |

**When your data "changes by itself", ask three questions:**

1. **Who else has a label on this object?** Check with `id()` or `is`.
2. **Is the object mutable?**
3. **Was it mutated, or was a label rebound?**

## 8. Test Yourself

Predict the output first, then click to check.

**Q1.**
```python
a = [1]
b = a
a = a + [2]
print(b)
```
<details><summary>Answer</summary>

`[1]`. `a + [2]` builds a new list and rebinds only `a`.
</details>

**Q2.**
```python
a = [1]
b = a
a += [2]
print(b)
```
<details><summary>Answer</summary>

`[1, 2]`. `+=` on a list mutates the shared object.
</details>

**Q3.**
```python
def reset(d):
    d = {}

data = {"x": 1}
reset(data)
print(data)
```
<details><summary>Answer</summary>

`{'x': 1}`. `d = {}` only rebinds the local label.
</details>

**Q4.**
```python
def reset(d):
    d.clear()

data = {"x": 1}
reset(data)
print(data)
```
<details><summary>Answer</summary>

`{}`. `.clear()` mutates the object that both labels share.
</details>

**Q5.** Can the value of `t = (1, [2])` ever print differently?
<details><summary>Answer</summary>

Yes. `t[1].append(3)` makes it `(1, [2, 3])`. The tuple's slots are fixed, but the list inside them is still mutable.
</details>

**Q6.** What's wrong with `def add_item(item, bucket=[]):`, and how do you fix it?
<details><summary>Answer</summary>

The default list is created once, when the function is defined, and every call shares it. Use `bucket=None`, then `if bucket is None: bucket = []`.
</details>